In [25]:
from groq import Groq

In [26]:
import os
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print("Groq Connected")


GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [6]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response.choices[0].message.content)

Hello. How can I assist you today?


In [7]:
def build_context(reranked_results):

    context = ""

    for result in reranked_results:

        paper = result["chunk"]["paper_title"]

        text = result["chunk"]["text"]

        context += f"\n\nSOURCE: {paper}\n"

        context += text

    return context

In [8]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__session__', '_i', '_ii', '_iii', '_i1', '_i2', 'Groq', '_i3', 'client', '_i4', '_i5', '_i6', 'response', '_i7', 'build_context', '_i8'])


In [9]:
import os

for root, dirs, files in os.walk("../data"):
    for file in files:
        print(os.path.join(root, file))

../data\factor_database.csv
../data\hybrid_candidates.pkl
../data\processed_papers\chunked_papers.json
../data\processed_papers\embedded_chunks.json
../data\processed_papers\papers_metadata.json
../data\raw_papers\2508.15237v2 (1).pdf
../data\raw_papers\2605.21504v1.pdf
../data\raw_papers\2605.23953v1.pdf
../data\raw_papers\2605.23959v1.pdf
../data\raw_papers\2605.25894v1.pdf
../data\raw_papers\2605.27848v1.pdf
../data\raw_papers\2605.27945v1.pdf
../data\raw_papers\2605.27977v1.pdf
../data\raw_papers\2605.28853v1.pdf
../data\raw_papers\2605.29413v1.pdf
../data\raw_papers\ssrn-5231379.pdf
../data\vector_store\faiss_index.bin


In [10]:
import json

with open(
    "../data/processed_papers/embedded_chunks.json",
    "r",
    encoding="utf-8"
) as f:
    chunks = json.load(f)

print(len(chunks))

643


In [11]:
print(chunks[0].keys())

dict_keys(['global_chunk_id', 'local_chunk_id', 'paper_title', 'filename', 'text', 'embedding'])


In [12]:
import faiss

index = faiss.read_index(
    "../data/vector_store/faiss_index.bin"
)

print(index.ntotal)

643


In [13]:
print(
    len(chunks),
    index.ntotal
)

643 643


In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding Model Loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding Model Loaded


In [15]:
import numpy as np

def retrieve(
    query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype(np.float32)

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            "score": float(score),
            "chunk": chunks[idx]
        })

    return results

In [16]:
results = retrieve(
    "stochastic volatility option pricing",
    top_k=5
)

for i, result in enumerate(results):

    print(
        i+1,
        result["chunk"]["paper_title"]
    )

1 Option pricing under non-Markovian stochastic volatility models: A deep
2 Option pricing under non-Markovian stochastic volatility models: A deep
3 Option pricing under non-Markovian stochastic volatility models: A deep
4 Option pricing under non-Markovian stochastic volatility models: A deep
5 Stochastic Volatility, Jumps, and Rates: A Unified Framework for


In [17]:
def build_context(results):

    context = ""

    for result in results:

        context += (
            f"\n\nSOURCE: {result['chunk']['paper_title']}\n"
        )

        context += result["chunk"]["text"]

    return context

In [18]:
context = build_context(results)

print(context[:2000])



SOURCE: Option pricing under non-Markovian stochastic volatility models: A deep
 stochastic diﬀerential equati on, allowing the application of standard analytical tools. We p ropose a deep signature approach for both linear and nonlinear representations and rigorously prove the conver gence of the algorithm. Numerical examples demonstrate the eﬀectiveness of our approach for both Markovian and non- Markovian volatility models, oﬀering a theoretically grounded and computationally eﬃcient framework for option pricing. 1 Introduction The pricing problem lies at the heart of mathematical ﬁnance , which is equivalent to computing an expectation such as E[Φ( ST )], (1) where Φ is a suﬃciently regular payoﬀ function and T denotes maturity. A typical choice of the underlying asset S is the stochastic volatility model as follows dSt =f (t,S t)vtdWt +g(t,S t)vtdBt, S 0 =s0, 0<t ≤ T, (2) ∗ School of Mathematics, Big Data Laboratory on Financial Sec urity and Behavior (Laboratory of Philosophy a

In [20]:
def generate_answer(query, context):

    prompt = f"""
    You are a quantitative finance research assistant.

    Answer the question using ONLY the context below.

    Question:
    {query}

    Context:
    {context}

    Answer:
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [21]:
query = "How are stochastic volatility models used in option pricing?"

context = build_context(results)

answer = generate_answer(
    query,
    context
)

print(answer)

Stochastic volatility models are used in option pricing by representing the underlying asset's dynamics as a stochastic differential equation (SDE) that incorporates both the asset price and its volatility. The volatility process is modeled as a separate stochastic process, which can be correlated with the asset price process. This allows for the estimation of option prices that depend not only on the current state of the asset but also on the entire historical path of the process.

In the context of option pricing, stochastic volatility models can be formulated as rough stochastic differential equations (RSDEs), which can be transformed into classical stochastic differential equations using linear or nonlinear combinations of time-extended Brownian motion signatures. This representation enables the application of standard analytical tools for option pricing.

The stochastic volatility models can be calibrated to market option data and interest-rate term structures, allowing for a syst

In [22]:
def ask_question(query, top_k=5):

    results = retrieve(
        query,
        top_k=top_k
    )

    context = build_context(results)

    answer = generate_answer(
        query,
        context
    )

    print("\nQUESTION:")
    print(query)

    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")
    print("=" * 60)

    seen = set()

    for result in results:

        title = result["chunk"]["paper_title"]

        if title not in seen:
            seen.add(title)
            print("-", title)

    return answer

In [23]:
ask_question(
    "How do stochastic volatility models improve option pricing?"
)


QUESTION:
How do stochastic volatility models improve option pricing?

ANSWER:
Stochastic volatility models improve option pricing by allowing for a more accurate representation of the underlying asset's price dynamics. They incorporate the volatility of the asset as a stochastic process, rather than assuming a constant volatility, which can lead to more precise estimates of option prices. This is particularly important for options with longer maturities or those that are deeply in-the-money or out-of-the-money, where the volatility of the underlying asset can have a significant impact on the option's value.

By accounting for the stochastic nature of volatility, these models can capture the complexities of real-world financial markets, such as the clustering of volatility and the leverage effect, which can lead to more accurate option pricing. Additionally, stochastic volatility models can be used to price options on assets with non-lognormal distributions, which can be important for

"Stochastic volatility models improve option pricing by allowing for a more accurate representation of the underlying asset's price dynamics. They incorporate the volatility of the asset as a stochastic process, rather than assuming a constant volatility, which can lead to more precise estimates of option prices. This is particularly important for options with longer maturities or those that are deeply in-the-money or out-of-the-money, where the volatility of the underlying asset can have a significant impact on the option's value.\n\nBy accounting for the stochastic nature of volatility, these models can capture the complexities of real-world financial markets, such as the clustering of volatility and the leverage effect, which can lead to more accurate option pricing. Additionally, stochastic volatility models can be used to price options on assets with non-lognormal distributions, which can be important for assets that exhibit fat-tailed or skewed distributions.\n\nOverall, the use 